# Confidence-Driven Inference Tutorial — Tutorial 1 (adaptive)

**Tutorial 1 (adaptive):** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonal-ssj/cdi-tutorial/blob/colab_support/tutorial_version_1_adaptive.ipynb)

**Setup:** run the cell below — on Colab it clones the repo and installs
dependencies; locally it installs into your selected venv. Then make sure your
API key is set (`.env` locally, or Colab Secrets). See the README for full details.

In [13]:
# === Setup — run this cell ===
# This cell installs dependencies, but it does NOT create a virtual environment.
# Locally it will only `pip install` once you're inside a venv: if no venv is
# detected it prints the terminal steps to do first, then you re-run this cell.
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

in_venv = sys.prefix != sys.base_prefix  # True when a virtual environment is active

if IN_COLAB:
    # Opened from GitHub, Colab only has this notebook — clone the repo to get
    # utils/, data/, and requirements, then work from inside it.
    if os.path.basename(os.getcwd()) != "cdi-tutorial":
        if not os.path.exists("cdi-tutorial"):
            # TODO: drop "-b colab_support" after merging this branch into main
            !git clone -b colab_support https://github.com/sonal-ssj/cdi-tutorial.git
        %cd cdi-tutorial
    # Use the minimal, loosely-pinned requirements so they resolve against
    # Colab's preinstalled packages (the frozen requirements.txt conflicts).
    !pip install -r requirements-core.txt
    print(
        "Repo cloned and dependencies installed (Colab).\n"
        "\nNext steps for Colab:\n"
        "  1. Open the Secrets panel (key icon in the left sidebar).\n"
        "  2. Add a secret named GROQ_API_KEY (or OPENAI_API_KEY) and paste your key.\n"
        "  3. Toggle 'Notebook access' on for that secret.\n"
        "  4. Run the cells below top-to-bottom."
    )
elif in_venv:
    !pip install -q -r requirements.txt
    print(
        "Virtual environment detected — dependencies installed into it.\n"
        "\nRemaining steps:\n"
        "  1. cp .env.example .env  ->  edit .env and set GROQ_API_KEY (or OPENAI_API_KEY).\n"
        "  2. Run the cells below top-to-bottom."
    )
else:
    print(
        "No virtual environment detected — skipping pip install so nothing is\n"
        "installed into your base Python. Do the TERMINAL steps first, then re-run:\n"
        "  1. (terminal)  python -m venv venv          # creates the virtual environment\n"
        "  2. (terminal)  source venv/bin/activate     # Windows: venv\\Scripts\\activate\n"
        "  3. In VSCode, select ./venv as the kernel (top-right kernel picker).\n"
        "  4. Re-run this cell  ->  installs requirements.txt into the venv.\n"
        "  5. cp .env.example .env  ->  edit .env and set GROQ_API_KEY (or OPENAI_API_KEY).\n"
        "  6. Run the cells below top-to-bottom."
    )

Repo cloned and dependencies installed (Colab).

Next steps for Colab:
  1. Open the Secrets panel (key icon in the left sidebar).
  2. Add a secret named GROQ_API_KEY (or OPENAI_API_KEY) and paste your key.
  3. Toggle 'Notebook access' on for that secret.
  4. Run the cells below top-to-bottom.


### Import libraries

In [14]:
import numpy as np
from scipy.stats import norm, bernoulli
import pprint
import pandas as pd
from tqdm import tqdm
from tqdm.notebook import tqdm
import time
from ppi_py.utils import bootstrap
import re
import openai
import requests
import json
from datetime import datetime, timezone
import time
import zipfile
import io
import random
from sklearn.linear_model import LogisticRegression
import os

# autoreload is a local-dev convenience (re-imports utils/*.py when you edit them).
# Load it defensively: Colab's older IPython on Python 3.12 crashes here because
# its autoreload imports the removed `imp` module, so skip it gracefully there.
try:
    _ip = get_ipython()
    _ip.run_line_magic("load_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")
except Exception as _e:
    print(f"Skipping autoreload ({_e}).")

from utils import llms, qualtrics, prolific, mturk, inference
from utils.llms import annotate_texts_with_llm, collect_llm_confidence, get_llm_annotations, get_client
from utils.config import load_llm_api_key, load_credential
from utils.qualtrics import create_and_activate_surveys
from utils.prolific import run_prolific_annotation_pipeline
from utils.mturk import run_mturk_annotation_pipeline
from utils.inference import train_sampling_rule, sampling_rule_predict, confidence_driven_inference, collect_initial_human_annotations, run_adaptive_sampling

Skipping autoreload (No module named 'imp').


### Setup credentials

In [15]:
# Human-annotation credentials (AWS / Qualtrics / Prolific). Only needed if
# COLLECT_HUMAN = True. Each key is resolved via Colab Secrets -> .env ->
# credentials.txt (see utils/config.py). Missing keys come back as None, which
# is fine when COLLECT_HUMAN = False.
AWS_ACCESS_KEY_ID = load_credential("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = load_credential("AWS_SECRET_ACCESS_KEY")
QUALTRICS_API_KEY = load_credential("QUALTRICS_API_KEY")
QUALTRICS_API_URL = load_credential("QUALTRICS_API_URL")
PROLIFIC_API_KEY = load_credential("PROLIFIC_API_KEY")

### Set parameters for Confidence-Driven Inference (CDI)

In [ ]:
# if true, we collect LLM annotations and human annotations from scratch, if false, load pre-collected ones
COLLECT_LLM = True   # set True to call the LLM (Groq/OpenAI) live; False loads pre-collected gpt-4o labels
COLLECT_HUMAN = False  # False = use pre-collected human labels from the dataset (no Prolific/MTurk API needed)

# if COLLECT_HUMAN = True, specify whether to use "Prolific" or "MTURK"
HUMAN_SOURCE = "MTURK"

alpha = 0.1  # desired error level for confidence interval
burnin_steps = 5  # we collect the first burnin_steps points to initialize sampling rule

n_batches = 2

n_human = 15 # budget on number of human annotations (including burnin_steps)

N = 100 # corpus size, or the size of the random subset of the corpus that will be annotated with an LLM

random_state = 42

#define the estimator function
#mask for valid labels and specify how to use weights
def mean_estimator(y, weights):
    y, weights = y[~np.isnan(y)], weights[~np.isnan(y)]
    return np.sum(y * weights) / np.sum(weights)

# Load the text corpus; we need two columns: texts and the text-based feature we will use for inference
# Text-based feature used for inference in this example is the presence of hedging, stored in "Feature_3" column
text_based_feature = 'Feature_3'

df = pd.read_csv('data/politeness_dataset.csv')[['Text',text_based_feature]]
df = df.sample(n = N, random_state = random_state)

# When COLLECT_HUMAN = False, Steps 2 & 3 use these pre-collected human labels
# instead of calling Prolific/MTurk (no human-annotation API access needed).
# Loaded here so Step 3 works even if Step 2 is resumed from a checkpoint
# (resuming skips Step 2's function, which would otherwise add this column).
if not COLLECT_HUMAN:
    df['Prediction_human'] = pd.read_csv('data/politeness_dataset.csv').sample(
        n=N, random_state=random_state)['Prediction_human'].values

### Set parameters for LLM annotation

In [17]:
# Choose the LLM backend. Switching providers is the only change needed here:
# the model and API key follow automatically.
PROVIDER = "groq"   # "groq" or "openai"

MODEL_BY_PROVIDER = {
    "groq":   "llama-3.1-8b-instant",        # alts: llama-3.3-70b-versatile, openai/gpt-oss-120b
    "openai": "gpt-4o-mini-2024-07-18",
}
# Groq deprecates models often. If a call fails with an invalid-model error,
# run get_client(PROVIDER, LLM_API_KEY).models.list() to see what's live.
model = MODEL_BY_PROVIDER[PROVIDER]

# Resolve the API key: Colab Secrets -> .env -> credentials.txt
LLM_API_KEY = load_llm_api_key(PROVIDER)

prompt = """Is the following text polite? Output either A or B. Output a letter only.
A) Polite
B) Impolite

Text: """

#category numerically coded as 1
positive_class = 'Polite'

mapping_categories = {
"A": "Polite",
"B": "Impolite"
}

sleep = 0.1
temperature = 0

llm_parameters = {
    "model": model,
    "prompt": prompt,
    "mapping_categories": mapping_categories,
    "sleep": sleep,
    "temperature": temperature,
    "LLM_API_KEY": LLM_API_KEY,
    "provider": PROVIDER,
    "positive_class": positive_class
}

### Set parameters for human annotation

In [18]:
## Task parameters
task_title = "Is the following text polite?"
annotation_instruction = "Is the following text polite?"
task_description = "Please annotate the politeness of the provided text. This is a real-time study where we're hoping to get immediate annotations."
categories = [
    "Polite",
    "Impolite"
]

## Additional Prolific settings
BASE_URL = "https://api.prolific.com/api/v1"
HEADERS = {
    "Authorization": f"Token {PROLIFIC_API_KEY}",
    "Content-Type": "application/json",
}
reward = 80
estimated_time = 1
maximum_allowed_time = 30 # How long a single annotator can take
BATCH_TIMEOUT = 30 # How long we'll wait for the resonses (in minutes) before moving on (replaces not collected annotations with np.nan)

## Additional MTURK settings
task_reward = '0.8'
minimum_approval_rate = 99 # >98% reccommended
minimum_tasks_approved = 0 # >5000 reccommended
annotation_instructions = {"question": task_description,
    "options": set(categories)}

human_annotation_parameters = {
    "categories": categories,
    "annotation_instruction": annotation_instruction,
    "annotation_instructions": annotation_instructions,
    "QUALTRICS_API_URL": QUALTRICS_API_URL,
    "QUALTRICS_API_KEY": QUALTRICS_API_KEY,
    "task_title": task_title,
    "task_description": task_description,
    "reward": reward,
    "estimated_time": estimated_time,
    "maximum_allowed_time": maximum_allowed_time,
    "HEADERS": HEADERS,
    "BASE_URL": BASE_URL,
    "BATCH_TIMEOUT": BATCH_TIMEOUT,
    "task_reward": task_reward,
    "minimum_approval_rate": minimum_approval_rate,
    "minimum_tasks_approved": minimum_tasks_approved,
    "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY_ID,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
    "positive_class": positive_class
}

### Checkpointing (resume without re-collecting)

Each step below saves the working `data` frame to a CSV in `checkpoints/`.
Because the collection steps make external API calls (LLM + human), re-running a
cell would otherwise overwrite `data` and lose those results.

- **`RESUME = True`** (default): if a step's checkpoint already exists, it is
  loaded and the (expensive) collection is skipped.
- Set **`RESUME = False`** to force a clean run and overwrite the checkpoints.
- To restart a single step from scratch, delete its file in `checkpoints/`.

In [19]:
# If True, reuse existing checkpoints instead of re-collecting. Set False for a clean run.
RESUME = True

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def _checkpoint_path(name):
    return os.path.join(CHECKPOINT_DIR, f"{name}.csv")

def has_checkpoint(name):
    return os.path.exists(_checkpoint_path(name))

def save_checkpoint(data, name):
    path = _checkpoint_path(name)
    data.to_csv(path, index=False)
    print(f"Saved checkpoint: {path}")
    return data

def load_checkpoint(name):
    path = _checkpoint_path(name)
    data = pd.read_csv(path)
    print(f"Resumed from checkpoint: {path}")
    return data

### Step 1: Collect LLM annotations for all the texts

In [20]:
if RESUME and has_checkpoint("step1_llm"):
    data = load_checkpoint("step1_llm")
else:
    data = get_llm_annotations(df=df,
        text_based_feature=text_based_feature,
        COLLECT_LLM=COLLECT_LLM,
        llm_parameters = llm_parameters,
        N=N,
        random_state=random_state)
    save_checkpoint(data, "step1_llm")

Resumed from checkpoint: checkpoints/step1_llm.csv


### Step 2: Collect warmup human annotations (initial set)

In [21]:
if RESUME and has_checkpoint("step2_human_warmup"):
    data = load_checkpoint("step2_human_warmup")
else:
    data = collect_initial_human_annotations(
        data=data,
        df=df,
        burnin_steps=burnin_steps,
        COLLECT_HUMAN=COLLECT_HUMAN,
        HUMAN_SOURCE=HUMAN_SOURCE,
        N=N,
        random_state=random_state,
        human_annotation_parameters = human_annotation_parameters)
    save_checkpoint(data, "step2_human_warmup")

Saved checkpoint: checkpoints/step2_human_warmup.csv


### Step 3: Strategically collect human annotations

In [22]:
if RESUME and has_checkpoint("step3_adaptive"):
    data = load_checkpoint("step3_adaptive")
else:
    data = run_adaptive_sampling(
        data=data,
        df=df,
        burnin_steps=burnin_steps,
        n_human=n_human,
        n_batches=n_batches,
        COLLECT_HUMAN=COLLECT_HUMAN,
        HUMAN_SOURCE=HUMAN_SOURCE,
        human_annotation_parameters = human_annotation_parameters)
    save_checkpoint(data, "step3_adaptive")



13 human datapoints collected in total.
Saved checkpoint: checkpoints/step3_adaptive.csv


### Step 4: Compute the CDI estimate and confidence intervals

We showcase estimation of $\mathrm{mean}(H)$: prevalence of the politeness, i.e., the fraction of texts in the corpus that are polite

In [23]:
estimate, (lower_bound, upper_bound) = confidence_driven_inference(
    estimator = mean_estimator,
    Y = data['human'].values,
    Yhat = data['llm'].values,
    sampling_probs =  data['sampling_probs'].values,
    sampling_decisions = data['sampling_decisions'].values,
    alpha = alpha)

print("CDI estimate of the target statistic (mean(H): prevalence of politeness):")
print('point estimate:',estimate.round(4))
print('confidence intervals:', lower_bound.round(4), upper_bound.round(4))

print("Ground truth mean(H) estimate (if we had access to human annotations on the full text corpus):")
print(np.mean(pd.read_csv('data/politeness_dataset.csv').sample(n = N, random_state = random_state)['Prediction_human'].values))

CDI estimate of the target statistic (mean(H): prevalence of politeness):
point estimate: 0.5144
confidence intervals: 0.2547 0.7511
Ground truth mean(H) estimate (if we had access to human annotations on the full text corpus):
0.58
